# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaKhanBUIC/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook constructs the production **Feature Vector** for the FlyRank Content Refresh prioritization model and executes an aggressive **Leakage and Privacy Audit**. We document every feature's temporal availability, handle categorical encoding and structured missingness safely, and deliberately stress-test our training pipeline against label leakage and group contamination.

## 1. Build the feature vector

Below, we engineer the complete feature matrix $\mathbf{X}$ from observable telemetry signals. We implement indicator flags for structured missingness and encode categorical dimensions cleanly:

In [1]:
# Feature Vector Construction Pipeline
import os, pandas as pd, numpy as np

csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/raw/content_refresh_anonymized.csv' if os.path.exists('../data/raw/content_refresh_anonymized.csv') else 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(csv_path)

# 1. Define Target Label (Strictly derived, never an input feature)
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 2. Missingness Indicator Flags (Safe handling of structured missingness)
df['has_keyword_data'] = df['search_volume'].notnull().astype(int)
df['has_valid_position'] = (df['avg_position'] > 0).astype(int)
df['has_word_count'] = df['word_count'].notnull().astype(int)

# 3. Numeric Pre-Decision Features
numeric_features = [
    'content_age_days', 'days_since_last_update', 'impressions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'search_volume', 'competition', 'cpc', 'word_count',
    'has_keyword_data', 'has_valid_position', 'has_word_count'
]

# 4. Categorical Encoding (One-Hot)
cat_cols = ['content_type', 'position_tier']
encoded_cats = pd.get_dummies(df[cat_cols], drop_first=True, dtype=int)

# 5. Imputation for numeric features (Zero fill for absent volume/metrics)
X_numeric = df[numeric_features].fillna(0)
X = pd.concat([X_numeric, encoded_cats], axis=1)
y = df['is_declining_label'].values

print('Feature Vector Matrix Built Successfully:')
print(f'- Total Rows: {X.shape[0]:,}')
print(f'- Total Feature Columns: {X.shape[1]}')
print(f'- Feature Names: {list(X.columns)}')


Feature Vector Matrix Built Successfully:
- Total Rows: 30,000
- Total Feature Columns: 20
- Feature Names: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'search_volume', 'competition', 'cpc', 'word_count', 'has_keyword_data', 'has_valid_position', 'has_word_count', 'content_type_feedly article', 'content_type_keyword article', 'position_tier_page_1', 'position_tier_page_3_5', 'position_tier_striking', 'position_tier_top_3']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature Name | Signal Meaning | Missingness Handling | Temporal Provenance (Available When?) |
|:---|:---|:---|:---|
| `content_age_days` | Total days elapsed since publication | Never null ($0$) | Pre-decision metadata |
| `days_since_last_update` | Staleness metric (days since last edit) | Never null ($0$) | Pre-decision metadata |
| `impressions_90d` | Trailing 90-day search visibility | Filled with $0$ | Observable at prediction cutoff |
| `avg_position` | Trailing average Google ranking rank | $0$ sentinel coded with flag | Observable at prediction cutoff |
| `ctr` | Trailing click-through rate percentage | Filled with $0.0$ | Observable at prediction cutoff |
| `engagement_rate` | GA4 on-page user engagement rate % | Filled with $0.0$ | Observable at prediction cutoff |
| `scroll_rate` | Content scroll depth percentage | Filled with $0.0$ | Observable at prediction cutoff |
| `has_keyword_data` | Binary indicator for keyword metadata | Explicit $0/1$ flag | Preserves category-specific missingness signal |
| `content_type_*` | Archetype (Article, Feedly, Comparison) | One-hot encoded | Pre-decision categorization |

In [2]:
# Feature Summary Statistics & Validation
print('Feature Matrix Summary Statistics (First 6 Features):')
print(X.iloc[:, :6].describe().round(2).T[['min', 'mean', '50%', 'max']])

# Verify Zero Inf/NaN values in Feature Matrix
assert not X.isnull().any().any(), 'Fatal: NaN values remain in feature matrix'
assert not np.isinf(X.values).any(), 'Fatal: Infinite values detected in feature matrix'
print('\n✓ Feature matrix clean: zero NaNs, zero infinities.')


Feature Matrix Summary Statistics (First 6 Features):
                         min     mean     50%       max
content_age_days        90.0   256.17  236.00     564.0
days_since_last_update   1.0    46.10   20.00     373.0
impressions_90d          1.0  5200.37  731.00  517715.0
avg_position             0.0    16.34   10.80     245.0
ctr                      0.0     0.51    0.07     100.0
engagement_rate          0.0     2.53    0.00     100.0

✓ Feature matrix clean: zero NaNs, zero infinities.


## 3. The leakage hunt

**Attacking Our Own Model (The Leakage Confession Test):**  
To prove that our honest pipeline is leak-free, we perform an intentional adversarial test: we train a Decision Tree model with our clean feature set, and compare it against an intentionally corrupted model that has access to the forbidden `trend_pct` column.

If a model exhibits near $1.000$ Precision@50, it is symptomatic of label leakage (cheating on the exam). An honest model must exhibit realistic, bounded precision.

In [3]:
# Adversarial Leakage Attack Test
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Honest Client-Holdout Split (Grouped by client_id to prevent memorization)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

# 1. Train Honest Model
honest_tree = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
honest_tree.fit(X.iloc[train_idx], y[train_idx])
honest_scores = honest_tree.predict_proba(X.iloc[test_idx])[:, 1]
honest_p50 = precision_at_k(honest_scores, y[test_idx], 50)

# 2. Train Deliberately Leaky Model (Injected with trend_pct)
X_leaky = X.copy()
X_leaky['trend_pct_LEAK'] = df['trend_pct'].fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
leaky_tree.fit(X_leaky.iloc[train_idx], y[train_idx])
leaky_scores = leaky_tree.predict_proba(X_leaky.iloc[test_idx])[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y[test_idx], 50)

print('=== Leakage Attack Test Results ===')
print(f'- Honest Model Test Precision@50: {honest_p50:.3f} (Realistic, honest predictive signal)')
print(f'- Leaky Model Test Precision@50:  {leaky_p50:.3f} (Suspicious 1.000 -> Confesses label leakage)')
assert honest_p50 < 0.99, 'Warning: Honest model score is suspiciously high!'
print('✓ Leakage audit passed: Honest features respect information boundaries.')


=== Leakage Attack Test Results ===
- Honest Model Test Precision@50: 0.680 (Realistic, honest predictive signal)
- Leaky Model Test Precision@50:  1.000 (Suspicious 1.000 -> Confesses label leakage)
✓ Leakage audit passed: Honest features respect information boundaries.


## 4. What I excluded and why

**Explicit Registry of Excluded Features:**

1. **`trend_pct` & `trend_direction`:**  
   * *Reason:* These fields literally define the target variable `is_declining_label`. Including them leaks the answer directly into the input space.
2. **`client_id` & `content_id`:**  
   * *Reason:* Identifiers encode tenant identities. Including them would allow models to memorize specific client domains instead of generalizable search ranking signals.
3. **`health_score` / Product Heuristics:**  
   * *Reason:* Proprietary health scores encode existing rule-based decisions, creating circular logic where the model merely learns to mimic human heuristics.

In [4]:
# Formal Exclusion Audit Query
forbidden_cols = ['trend_pct', 'trend_direction', 'client_id', 'content_id']
active_feature_set = set(X.columns)

print('Exclusion Registry Audit:')
for col in forbidden_cols:
    in_features = col in active_feature_set
    status = '✗ VIOLATION' if in_features else '✓ EXCLUDED (Safe)'
    print(f'- {col:20s}: {status}')
    assert not in_features, f'Security Breach: {col} is in the active feature matrix!'

print('\n✓ Privacy and Leakage Policy: 100% Compliant.')


Exclusion Registry Audit:
- trend_pct           : ✓ EXCLUDED (Safe)
- trend_direction     : ✓ EXCLUDED (Safe)
- client_id           : ✓ EXCLUDED (Safe)
- content_id          : ✓ EXCLUDED (Safe)

✓ Privacy and Leakage Policy: 100% Compliant.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.